In [5]:
import pandas as pd
import numpy as np

In [6]:
df = pd.read_csv('data/results.csv', parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)
df


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False
...,...,...,...,...,...,...,...,...,...
49282,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True
49283,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True
49284,2026-06-27,Algeria,Austria,NaN,NaN,FIFA World Cup,Kansas City,United States,True
49285,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True


In [7]:
def calcular_resultado(row):
    if row['home_score'] > row['away_score']:
        return 1
    elif row['home_score'] < row['away_score']:
        return 2
    else:
        return 0

df['resultado'] = df.apply(calcular_resultado, axis=1)
df

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,resultado
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,1
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1
...,...,...,...,...,...,...,...,...,...,...
49282,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True,0
49283,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True,0
49284,2026-06-27,Algeria,Austria,NaN,NaN,FIFA World Cup,Kansas City,United States,True,0
49285,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True,0


In [8]:
df['es_neutral']= df['neutral'].astype(int)
df

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,resultado,es_neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,0,0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,1,0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1,0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,0,0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1,0
...,...,...,...,...,...,...,...,...,...,...,...
49282,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True,0,1
49283,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True,0,1
49284,2026-06-27,Algeria,Austria,NaN,NaN,FIFA World Cup,Kansas City,United States,True,0,1
49285,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True,0,1


In [12]:
# Asignamos un peso del 1 al 4 según la relevancia competitiva
def mapear_importancia_torneo(torneo):
    torneo = str(torneo)
    if 'FIFA World Cup' in torneo and 'FIFA World Cup qualification' not in torneo:
        return 4  # El Mundial absoluto
    elif 'Copa América' in torneo or 'UEFA Euro' in torneo and 'qualification' not in torneo:
        return 3  # Torneos continentales mayores
    elif 'FIFA World Cup qualification' in torneo or 'nations league' in torneo or 'confederations' in torneo:
        return 2  # Clasificatorios y torneos oficiales menores
    elif 'friendly' in torneo:
        return 1  # Amistosos
    else:
        return 1.5 # Otros torneos menores (Copas de invitación, etc.)

df['importancia_torneo'] = df['tournament'].apply(mapear_importancia_torneo)
df

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,resultado,es_neutral,importancia_torneo
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,0,0,1.5
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,1,0,1.5
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1,0,1.5
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,0,0,1.5
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1,0,1.5
...,...,...,...,...,...,...,...,...,...,...,...,...
49282,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True,0,1,4.0
49283,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True,0,1,4.0
49284,2026-06-27,Algeria,Austria,NaN,NaN,FIFA World Cup,Kansas City,United States,True,0,1,4.0
49285,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True,0,1,4.0


In [13]:
import pandas as pd
import numpy as np


# =================================================================
# PASO 2: Construir la Línea de Tiempo Unificada (Locales y Visitantes)
# =================================================================
# Extraemos los datos del equipo LOCAL
df_local = df[['date', 'home_team', 'home_score', 'away_score', 'resultado']].rename(
    columns={'home_team': 'equipo', 'home_score': 'goles_favor', 'away_score': 'goles_contra'}
)
df_local['puntos'] = df_local['resultado'].map({1: 3, 0: 1, 2: 0})

# Extraemos los datos del equipo VISITANTE
df_visitante = df[['date', 'away_team', 'away_score', 'home_score', 'resultado']].rename(
    columns={'away_team': 'equipo', 'away_score': 'goles_favor', 'home_score': 'goles_contra'}
)
df_visitante['puntos'] = df_visitante['resultado'].map({1: 0, 0: 1, 2: 3})

# Juntamos ambos conjuntos cronológicamente
df_linea_tiempo = pd.concat([df_local, df_visitante]).sort_values('date')
groupby_equipo = df_linea_tiempo.groupby('equipo')

# =================================================================
# PASO 3: Calcular Medias Móviles Cronológicas (Últimos 10 partidos)
# =================================================================
# Usamos .shift(1) para evitar fuga de datos (el partido actual no sabe sus propios goles antes de jugar)
df_linea_tiempo['goles_ultimos_10'] = groupby_equipo['goles_favor'].shift(1).rolling(10, min_periods=1).mean()
df_linea_tiempo['goles_concedidos_ultimos_10'] = groupby_equipo['goles_contra'].shift(1).rolling(10, min_periods=1).mean()
df_linea_tiempo['forma_ultimos_10'] = groupby_equipo['puntos'].shift(1).rolling(10, min_periods=1).mean()

# Limpiamos filas duplicadas para quedarnos con un registro único por fecha y equipo
df_linea_limpia = df_linea_tiempo[['date', 'equipo', 'goles_ultimos_10', 'goles_concedidos_ultimos_10', 'forma_ultimos_10']].drop_duplicates(subset=['date', 'equipo'])

# =================================================================
# PASO 4: Unificar las Métricas Recientes al DataFrame Principal
# =================================================================
# Unir datos para el Equipo Local de forma limpia
df = df.merge(df_linea_limpia, left_on=['date', 'home_team'], right_on=['date', 'equipo'], how='left')
df = df.rename(columns={
    'goles_ultimos_10': 'media_goles_recientes_local', 
    'goles_concedidos_ultimos_10': 'media_goles_concedidos_local',
    'forma_ultimos_10': 'forma_reciente_local'
}).drop(columns=['equipo'])

# Unir datos para el Equipo Visitante de forma limpia
df = df.merge(df_linea_limpia, left_on=['date', 'away_team'], right_on=['date', 'equipo'], how='left')
df = df.rename(columns={
    'goles_ultimos_10': 'media_goles_recientes_visitante', 
    'goles_concedidos_ultimos_10': 'media_goles_concedidos_visitante',
    'forma_ultimos_10': 'forma_reciente_visitante'
}).drop(columns=['equipo'])

# =================================================================
# PASO 5: Calcular Historial y Peso de la Camiseta (Importancia 4)
# =================================================================
# 1. Calculamos los puntos base primero en el DataFrame general de manera segura
df['pts_loc'] = df.apply(lambda r: 3 if r['resultado'] == 1 else (1 if r['resultado'] == 0 else 0), axis=1)
df['pts_vis'] = df.apply(lambda r: 3 if r['resultado'] == 2 else (1 if r['resultado'] == 0 else 0), axis=1)

# 2. Filtramos para analizar solo la historia en la importancia 4 (FIFA World Cup)
partidos_mundial = df[df['importancia_torneo'] == 4.0]

pts_historicos_local = partidos_mundial.groupby('home_team')['pts_loc'].mean().to_dict()
pts_historicos_visitante = partidos_mundial.groupby('away_team')['pts_vis'].mean().to_dict()

df['peso_camiseta_local'] = df['home_team'].map(pts_historicos_local).fillna(1.0)
df['peso_camiseta_visitante'] = df['away_team'].map(pts_historicos_visitante).fillna(1.0)

# =================================================================
# PASO 6: Crear los Predictores Diferenciales Finales (Cálculo Limpio)
# =================================================================
# Al nacer de columnas únicas y renombradas, esta sección correrá perfectamente sin KeyErrors ni ValueErrors
df['dif_goles_recientes'] = df['media_goles_recientes_local'] - df['media_goles_recientes_visitante']
df['dif_forma_reciente'] = df['forma_reciente_local'] - df['forma_reciente_visitante']
df['dif_goles_concedidos'] = df['media_goles_concedidos_local'] - df['media_goles_concedidos_visitante']
df['dif_peso_camiseta'] = df['peso_camiseta_local'] - df['peso_camiseta_visitante']

# NUEVO TRUCO: Usamos directamente tu columna 'es_neutral' (que ya vale 1 o 0 en tu imagen)
# Invertimos el valor para crear la 'ventaja_localia' (Si es neutral es 0, si no es neutral es 1)
df['ventaja_localia'] = df['es_neutral'].apply(lambda x: 0 if x == 1 else 1)

# =================================================================
# PASO 7: Filtros de Era y Guardado de Datos Limpios
# =================================================================
# Filtramos por la era moderna (desde 2015) como tenías planeado
df_final = df[df['date'].dt.year >= 2010]

# Aplicamos tu restricción para dejar fuera los partidos futuros del Mundial 2026 en el set de entreno
df_final = df_final[df_final['date'] < '2026-05-01']

# Eliminamos nulos en las columnas esenciales de entrenamiento
columnas_control_nulos = ['resultado', 'media_goles_recientes_local', 'forma_reciente_local']
df_final = df_final.dropna(subset=columnas_control_nulos)

# Guardamos el archivo final limpio en la ruta protegida por tu .gitignore
df_final.to_csv('data/partidos_entrenamiento.csv', index=False)

print(f"¡Éxito rotundo! Archivo procesado. Filas finales guardadas: {len(df_final)}")


¡Éxito rotundo! Archivo procesado. Filas finales guardadas: 15632
